# Capstone Exercise: Berlin Bike Share Analysis

Congratulations! You've just joined a Berlin mobility startup as a data analyst. Your first task is to analyze the company's bike share data and prepare a stakeholder report. This exercise brings together everything you've learned about Python, NumPy, and Pandas.

## Learning Objectives

- Apply data loading and exploration techniques
- Use groupby, aggregation, and filtering
- Create meaningful visualizations
- Extract actionable insights from data
- Practice professional data analysis workflow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
# Load the data
bikes = pd.read_csv('../../data/berlin_bikes.csv')

# Convert datetime columns
bikes['start_time'] = pd.to_datetime(bikes['start_time'])
bikes['end_time'] = pd.to_datetime(bikes['end_time'])

print(f"Dataset shape: {bikes.shape}")
print(f"\nColumn dtypes:")
print(bikes.dtypes)
print(f"\nFirst few rows:")
bikes.head()

## Dataset Description

The dataset contains bike share trip records with the following columns:
- `duration`: Trip duration in seconds
- `start_time`: Trip start timestamp
- `end_time`: Trip end timestamp
- `start_station_id`: ID of the starting station
- `start_station`: Name of the starting station
- `end_station_id`: ID of the ending station
- `end_station`: Name of the ending station
- `bike_id`: Unique identifier for the bike
- `user_type`: Type of user (Casual or Member)

---

## Question 1: Dataset Overview

How many trips are in the dataset? What is the date range covered?

In [ ]:
# Calculate number of trips and date range
num_trips = len(bikes)
start_date = bikes['start_time'].min()
end_date = bikes['start_time'].max()
date_range_days = (end_date - start_date).days

print(f"Total trips: {num_trips:,}")
print(f"Date range: {start_date.date()} to {end_date.date()}")
print(f"Duration: {date_range_days} days")

---

## Question 2: Popular Stations

What are the top 5 most popular start stations?

In [ ]:
# Find top 5 start stations
top_stations = bikes['start_station'].value_counts().head(5)
print("Top 5 start stations:")
print(top_stations)

# Visualize
fig, ax = plt.subplots(figsize=(12, 5))
top_stations.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of Trips')
ax.set_ylabel('Station')
ax.set_title('Top 5 Most Popular Start Stations')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---

## Question 3: Trip Duration by User Type

What is the average trip duration by user type (Casual vs Member)?

In [ ]:
# Calculate average duration by user type (convert to minutes)
duration_by_user = bikes.groupby('user_type')['duration'].agg(['mean', 'median', 'count'])
duration_by_user['mean_minutes'] = duration_by_user['mean'] / 60
duration_by_user['median_minutes'] = duration_by_user['median'] / 60

print("Trip duration by user type:")
print(duration_by_user.round(2))

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
duration_by_user['mean_minutes'].plot(kind='bar', ax=ax, color=['#e74c3c', '#3498db'])
ax.set_ylabel('Average Duration (minutes)')
ax.set_xlabel('User Type')
ax.set_title('Average Trip Duration by User Type')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

---

## Question 4: Daily Pattern

Plot the number of trips per day of the week.

In [ ]:
# Trips by day of week
bikes['dayofweek'] = bikes['start_time'].dt.dayofweek
bikes['day_name'] = bikes['start_time'].dt.day_name()

trips_by_day = bikes.groupby('dayofweek').size()
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(day_names, trips_by_day.values, color='steelblue')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Number of Trips')
ax.set_title('Bike Trips by Day of Week')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---

## Question 5: Weekday vs Weekend

Is there a difference in trip duration between weekdays and weekends? Support your answer with a visualization.

In [ ]:
# Compare weekday vs weekend
bikes['is_weekend'] = bikes['dayofweek'] >= 5
bikes['day_type'] = np.where(bikes['is_weekend'], 'Weekend', 'Weekday')
bikes['duration_minutes'] = bikes['duration'] / 60

# Statistics
weekend_comparison = bikes.groupby('day_type')['duration_minutes'].agg(['mean', 'median', 'std', 'count'])
print("Duration comparison (minutes):")
print(weekend_comparison.round(2))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
ax1 = axes[0]
bikes_sample = bikes.sample(min(10000, len(bikes)), random_state=42)  # Sample for box plot
bikes_sample.boxplot(column='duration_minutes', by='day_type', ax=ax1)
ax1.set_title('Trip Duration Distribution')
ax1.set_xlabel('Day Type')
ax1.set_ylabel('Duration (minutes)')
ax1.set_ylim(0, 60)  # Focus on typical trip lengths
plt.suptitle('')  # Remove automatic title

# Bar chart of means
ax2 = axes[1]
weekend_comparison['mean'].plot(kind='bar', ax=ax2, color=['#3498db', '#e74c3c'])
ax2.set_ylabel('Average Duration (minutes)')
ax2.set_title('Average Trip Duration: Weekday vs Weekend')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

# Statistical insight
diff = weekend_comparison.loc['Weekend', 'mean'] - weekend_comparison.loc['Weekday', 'mean']
print(f"\nWeekend trips are on average {diff:.1f} minutes {'longer' if diff > 0 else 'shorter'} than weekday trips.")

---

## Question 6: Popular Routes

Find the top 10 most common station-to-station routes.

In [ ]:
# Create route column
bikes['route'] = bikes['start_station'] + ' → ' + bikes['end_station']

# Top 10 routes
top_routes = bikes['route'].value_counts().head(10)

print("Top 10 most popular routes:")
for i, (route, count) in enumerate(top_routes.items(), 1):
    print(f"{i:2d}. {route}: {count:,} trips")

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
top_routes.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of Trips')
ax.set_ylabel('Route')
ax.set_title('Top 10 Most Popular Routes')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---

## Question 7: Peak Hours

Which hour of the day has peak usage? Does this differ between user types?

In [ ]:
# Trips by hour and user type
bikes['hour'] = bikes['start_time'].dt.hour

hourly_by_user = bikes.groupby(['hour', 'user_type']).size().unstack(fill_value=0)

print("Peak hours by user type:")
for user_type in hourly_by_user.columns:
    peak_hour = hourly_by_user[user_type].idxmax()
    peak_trips = hourly_by_user[user_type].max()
    print(f"  {user_type}: {peak_hour}:00 ({peak_trips:,} trips)")

# Visualize
fig, ax = plt.subplots(figsize=(14, 6))

hourly_by_user.plot(ax=ax, marker='o')

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Trips')
ax.set_title('Hourly Trip Pattern by User Type')
ax.set_xticks(range(24))
ax.legend(title='User Type')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Question 8: Additional Insight

What is one additional insight you found that would be interesting for a stakeholder? Show it with a visualization.

In [ ]:
# Example: Bike utilization analysis
# Which bikes are most heavily used?

bike_usage = bikes.groupby('bike_id').agg(
    total_trips=('duration', 'count'),
    total_duration_hours=('duration', lambda x: x.sum() / 3600),
    avg_trip_minutes=('duration', lambda x: x.mean() / 60)
).round(2)

# Distribution of trips per bike
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of trips per bike
ax1 = axes[0]
ax1.hist(bike_usage['total_trips'], bins=50, edgecolor='white', alpha=0.7)
ax1.set_xlabel('Number of Trips')
ax1.set_ylabel('Number of Bikes')
ax1.set_title('Distribution of Trips per Bike')
ax1.axvline(bike_usage['total_trips'].median(), color='red', linestyle='--', label=f'Median: {bike_usage["total_trips"].median():.0f}')
ax1.legend()

# Top 10 most used bikes
ax2 = axes[1]
top_bikes = bike_usage.nlargest(10, 'total_trips')['total_trips']
top_bikes.plot(kind='bar', ax=ax2, color='steelblue')
ax2.set_xlabel('Bike ID')
ax2.set_ylabel('Number of Trips')
ax2.set_title('Top 10 Most Used Bikes')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nInsight: Bike utilization varies significantly.")
print(f"  - Most used bike: {bike_usage['total_trips'].max():,.0f} trips")
print(f"  - Least used bike: {bike_usage['total_trips'].min():,.0f} trips")
print(f"  - Median usage: {bike_usage['total_trips'].median():,.0f} trips")
print("\nRecommendation: Consider rotating high-use bikes for maintenance and")
print("investigating why some bikes see low usage (location, condition, etc.)")

---

## Bonus Challenge: Merge Practice

Load the Titanic dataset and perform a hypothetical merge exercise. Create a lookup table of class amenities and join it with the passenger data.

In [ ]:
# Load Titanic data
titanic = pd.read_csv('../../data/titanic.csv')

# Create a lookup table
class_amenities = pd.DataFrame({
    'Pclass': [1, 2, 3],
    'deck_location': ['Upper', 'Middle', 'Lower'],
    'dining_area': ['Grand Saloon', 'Second Class Dining', 'General Mess'],
    'cabin_size_sqft': [200, 120, 60]
})

print("Amenities lookup:")
print(class_amenities)

# Merge
titanic_with_amenities = pd.merge(titanic, class_amenities, on='Pclass', how='left')

print("\nMerged data:")
titanic_with_amenities[['Name', 'Pclass', 'deck_location', 'dining_area', 'cabin_size_sqft']].head(10)

---

## Summary

Congratulations on completing the capstone exercise! You've demonstrated skills in:

- ✅ Data loading and inspection
- ✅ DateTime manipulation
- ✅ GroupBy and aggregation
- ✅ Data visualization
- ✅ Extracting business insights
- ✅ Merge operations

These are the foundational skills you'll use every day as a data analyst or data scientist.